# Risk-on vs Risk-off Regimes

Financial markets broad regimes that reflect investor behaviour and global liquidity conditions:

- **Risk-on** → investors seek return and take risk  
- **Risk-off** → investors seek safety and reduce risk  

These regimes drive cross-asset correlations, FX carry performance, credit spreads, and equity volatility.



### Economic intuition

Risk-on environments typically occur when:
- growth expectations improve  
- liquidity is abundant  
- central banks are supportive  
- financial stress is low  

Risk-off environments typically occur when:
- recession fears rise  
- financial conditions tighten  
- geopolitical risk increases  
- market volatility spikes  

Carry trades, credit markets and equity markets are strongly tied to these cycles.


### Ex.: change of regime impact on investment strategies

Lets take as an example **carry trade strategies**. These borrow in low-yield currencies and invest in high-yield assets, making them exposed to:

- FX risk  
- credit risk  
- liquidity risk  

Risk-off episodes trigger a **carry unwind**:

1. Investors sell risky assets  
2. Leveraged positions are reduced  
3. Funding currencies are repurchased  
4. High-yield currencies depreciate rapidly  

This feedback loop explains why FX, credit, equities and volatility often move together during stress events.
During risk-off episodes investors deleverage and unwind these positions, producing large and rapid cross-asset moves.


### Cross-asset behaviour

| Indicator | Interpretation |
|---|---|
| USD Index (DXY Broad) | Global USD liquidity and funding stress |
| VIX | Equity market volatility and demand for protection |
| IG OAS | Investment-grade corporate credit risk |
| HY OAS | High-yield corporate credit risk |
| GPR | Geopolitical risk |
| Brent Oil | Global growth and inflation expectations |
| 5Y Breakeven Inflation | Inflation expectations |
| 5Y Real Yield | Real interest rates and financial conditions |


### Typical regime behaviour

| Asset / Indicator | Risk-on | Risk-off |
|---|---|---|
| Global equities | Rising | Falling |
| Credit spreads (IG, HY) | Tightening | Widening |
| VIX | Low | Elevated |
| USD | Weakening | Strengthening |
| JPY / CHF | Weakening | Strengthening |
| EM FX | Appreciating | Depreciating |
| Oil | Rising | Falling |
| Breakeven inflation | Rising | Falling |
| Real yields | Stable / Rising slowly | Falling initially (flight to safety) |



In [ ]:
from quant_risk.setup import base, macro

np, pd, plt = base()
fed_client, store, external_store = macro()

In [ ]:
series = [
    "US10Y",
    "US2Y",
    "VIX",
    "IG_OAS",
    "HY_OAS",
    "BRENT",
    "BREAKEVEN5Y",
    "REAL5Y",
    "EURUSD",
]

df_fred = store.build_panel(series)
df_fred = store.align(df_fred)

df_fred["Curve_Slope"] = df_fred["US10Y"] - df_fred["US2Y"]
df_fred.head()

In [ ]:
df_fred.tail()

In [ ]:
gpr = external_store.get_gpr()
gpr.head()

In [ ]:
gold = external_store.get_yfinance("GOLD")
gold.head()

In [ ]:
df = df_fred.join(gold, how="outer").join(gpr, how="outer").ffill()
df.head()

In [ ]:


# =========================
# CHANGES
# =========================
df_chg = df.pct_change()

# =========================
# RISK EVENTS
# =========================
events = {
    "GFC": ("2008-09-01","2009-03-01"),
    "Euro": ("2011-08-01","2012-01-01"),
    "COVID": ("2020-02-15","2020-04-15"),
    "2022": ("2022-02-01","2022-10-01"),
}

def shade(ax):
    for _, (s,e) in events.items():
        ax.axvspan(pd.to_datetime(s), pd.to_datetime(e), alpha=0.15)

# =========================
# PLOTS - LEVELS
# =========================
fig, axes = plt.subplots(4,3, figsize=(12,10))
axes = axes.flatten()

for ax, col in zip(axes, df.columns):
    ax.plot(df.index, df[col])
    ax.set_title(col)
    shade(ax)

fig.suptitle("Macro Risk Dashboard - Levels")
plt.tight_layout()
plt.show()

# =========================
# PLOTS - CHANGES
# =========================
fig, axes = plt.subplots(4,3, figsize=(12,10))
axes = axes.flatten()

for ax, col in zip(axes, df_chg.columns):
    ax.plot(df_chg.index, df_chg[col])
    ax.set_title(col + " % change")
    shade(ax)

fig.suptitle("Macro Risk Dashboard - Changes")
plt.tight_layout()
plt.show()